In [ ]:
import numpy as np
import pandas as pd
import re
from pathlib import Path
import warnings
import yaml

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## Import data

In [ ]:
raw_data = pd.read_csv(dataset_config['path_stata'] + 'CNKI/CNKI_DID_not_standard.csv')
raw_data

In [ ]:
matching_info = pd.read_csv(dataset_config['path_cnki'] + 'CNKI_standard_Qichacha/matching_info.csv')
matching_info = matching_info.drop_duplicates(subset=['firm_name'])
matching_info

In [ ]:
corporate_info = pd.read_csv(dataset_config['path_cnki'] + 'CNKI_standard_Qichacha/corporate_info.csv', usecols=['firm_name_qcc', '所属集团', '企业（机构）类型', '统一社会信用代码', '所属行业'])
corporate_info = corporate_info.drop_duplicates(subset=['firm_name_qcc'])
corporate_info

In [ ]:
# Assuming corporate_info is your dataframe
unique_groups = corporate_info['所属集团'].nunique()

print(f"Number of unique groups in '所属集团': {unique_groups}")

## merge

In [ ]:
raw_data_with_qcc= pd.merge(raw_data, matching_info, on='firm_name', how='left')
raw_data_with_qcc

In [ ]:
nan_count = raw_data_with_qcc['firm_name_qcc'].isna().sum()
print(f'Number of NaN values in "firm_name_qcc": {nan_count}')

In [ ]:
raw_data_with_qcc_corporate = pd.merge(raw_data_with_qcc, corporate_info, on='firm_name_qcc', how='left')
raw_data_with_qcc_corporate.head()

In [ ]:
raw_data_with_qcc_corporate['corporate'] = raw_data_with_qcc_corporate['所属集团'].fillna(raw_data_with_qcc_corporate['firm_name'])
raw_data_with_qcc_corporate['corporate_id'] = pd.factorize(raw_data_with_qcc_corporate['corporate'])[0]
raw_data_with_qcc_corporate['SOE'] = raw_data_with_qcc_corporate['企业（机构）类型'].str.contains('国有', na=False).astype(int)
raw_data_with_qcc_corporate

In [ ]:
raw_data_with_qcc_corporate.to_csv(dataset_config['path_processed'] + 'CNKI/06_CNKI_DID.csv', index=False, encoding='utf-8')